# Nanoscope: permanent Kaggle runner

Import this example into a **permanent notebook** once. The local Kaggle CLI only
uploads new versions of `workspace.zip`; you run this notebook yourself with its
attached data, GPU, Internet and secrets settings. No kernel submission through the CLI.

Before running:
1. Attach your private workspace dataset and select its current version.
2. Enable Internet and select your GPU(s), for example T4 x2.
3. Create and enable `HF_TOKEN`, `HF_REPO_ID`, and `WANDB_API_KEY` secrets for this
   notebook. Use an existing private HF model repository with a write-capable token.
   `WANDB_ENTITY` is optional. Secret values belong in Kaggle's secrets UI.

The default is a **four-step toy-model smoke test** with fixture text, GPU training,
HF checkpoints, W&B logging, and periodic validation. It is not an M1 model or a
quality benchmark. First verify the uploaded checkpoints and evaluation results;
then select your own experiment configs in the next cell.

Run cells in order. Training is a subprocess so dependency installation does not
require importing/reloading PyTorch inside the notebook kernel.

In [ ]:
from pathlib import Path

# None auto-selects workspace.zip only when exactly one attachment contains it.
WORKSPACE_ZIP = None  # Or Path("/kaggle/input/your-dataset/workspace.zip")
PROJECT_BASE = Path("/kaggle/working/nanoscope-workspaces")
EXPORT_DIR = Path("/kaggle/working/nanoscope-exports")

TRAIN_CONFIG = "configs/eval/kaggle-smoke.yaml"
EVAL_CONFIG = "configs/eval/kaggle-validation.yaml"
CORPUS_CONFIG = "configs/eval/local-corpus.yaml"
# Set CORPUS_CONFIG = None when EVAL_CONFIG.corpus points at an already-prepared attachment.
# For training without evaluation, set EVAL_CONFIG = None and CORPUS_CONFIG = None.

RESUME = "auto"  # Starts a new run if absent; resumes the latest local checkpoint otherwise.
# Fresh session: "hf://OWNER/REPOSITORY/runs/eval-kaggle-smoke"
STOP_AFTER_STEP = None  # E.g. 2 to check interruption/recovery without changing max_steps.
INSTALL_DEPENDENCIES = True  # False only after installation in this same session.

## Extract the uploaded workspace

Each ZIP gets its own directory named by content hash, so updating the attached
workspace cannot leave stale source files mixed into a newer version. `@/` in your
YAML resolves to this extracted project root through its `pyproject.toml`.
Training outputs should use the stable `/kaggle/working/nanoscope-runs` path in the
Kaggle training config, so they remain available when you switch source archives.

In [ ]:
import hashlib
import json
import os
import shutil
import tempfile
import zipfile

if WORKSPACE_ZIP is None:
    candidates = sorted(Path("/kaggle/input").rglob("workspace.zip"))
    if len(candidates) != 1:
        raise RuntimeError(
            f"Set WORKSPACE_ZIP explicitly; found {len(candidates)} workspace.zip files"
        )
    workspace_zip = candidates[0]
else:
    workspace_zip = Path(WORKSPACE_ZIP)

hasher = hashlib.sha256()
with workspace_zip.open("rb") as handle:
    for chunk in iter(lambda: handle.read(1024 * 1024), b""):
        hasher.update(chunk)
workspace_hash = hasher.hexdigest()
PROJECT_BASE.mkdir(parents=True, exist_ok=True)
PROJECT = PROJECT_BASE / workspace_hash
if not PROJECT.exists():
    temporary = Path(tempfile.mkdtemp(prefix=".extract-", dir=PROJECT_BASE))
    try:
        with zipfile.ZipFile(workspace_zip) as archive:
            for entry in archive.infolist():
                target = (temporary / entry.filename).resolve()
                if not target.is_relative_to(temporary.resolve()):
                    raise ValueError("Workspace ZIP contains a path outside the project")
            archive.extractall(temporary)
        if not (temporary / "pyproject.toml").is_file():
            raise ValueError("workspace.zip must contain pyproject.toml at its root")
        (temporary / "workspace-provenance.json").write_text(
            json.dumps({"workspace_sha256": workspace_hash}, indent=2)
        )
        temporary.rename(PROJECT)
    finally:
        if temporary.exists():
            shutil.rmtree(temporary)
os.chdir(PROJECT)
for selected in (TRAIN_CONFIG, EVAL_CONFIG, CORPUS_CONFIG):
    if selected is not None and not (PROJECT / selected).is_file():
        raise FileNotFoundError(
            f"Config missing from workspace: {selected}; update the dataset version"
        )
print(f"Project: {PROJECT}\nWorkspace SHA-256: {workspace_hash}")

## Install dependencies and load enabled secrets

The hashed requirements preserve Kaggle's installed PyTorch. All training commands
use fresh subprocesses with the installed environment. The notebook does not need
SciPy/Matplotlib; generate comparison reports locally with `uv sync --extra eval`.

The secret cell retrieves enabled values through Kaggle's
[`UserSecretsClient.get_secret`](https://github.com/Kaggle/docker-python/blob/main/patches/kaggle_secrets.py).
It prints labels only, never values. Required secrets are selected from the training
config: HF credentials for required persistence, W&B credentials for online logging.

In [ ]:
import signal
import subprocess
import sys


def run_command(*arguments):
    """Stream output; notebook interrupt asks the training CLI to checkpoint and stop."""
    with subprocess.Popen(list(arguments), start_new_session=True) as child:
        try:
            status = child.wait()
        except KeyboardInterrupt:
            print("Stop requested. Waiting for the CLI to checkpoint and finish...")
            child.send_signal(signal.SIGTERM)
            status = child.wait()
        if status:
            raise subprocess.CalledProcessError(status, list(arguments))


if INSTALL_DEPENDENCIES:
    run_command(sys.executable, "-m", "pip", "install", "--require-hashes", "-r",
                str(PROJECT / "requirements-kaggle.lock"))
    run_command(sys.executable, "-m", "pip", "install", "--no-deps", "-e", str(PROJECT))

In [ ]:
# Read the resolved config in a fresh process; keep notebook imports lightweight.
config_data = json.loads(subprocess.check_output(
    [sys.executable, "-c",
     "import json,sys; from nanoscope.config import load_config; "
     "print(json.dumps(load_config(sys.argv[1]).to_dict()))", TRAIN_CONFIG], text=True
))
required = []
optional = []
if config_data["checkpoint"]["hub_policy"] == "required":
    required.append("HF_TOKEN")
    if not config_data["checkpoint"]["hub_repo"]:
        required.append("HF_REPO_ID")
elif config_data["checkpoint"]["hub_policy"] == "best_effort":
    optional.extend(["HF_TOKEN", "HF_REPO_ID"])
if str(RESUME).startswith("hf://") and "HF_TOKEN" not in required:
    required.append("HF_TOKEN")
if config_data["logging"]["wandb_mode"] == "online":
    required.append("WANDB_API_KEY")
    if not config_data["logging"]["entity"]:
        optional.append("WANDB_ENTITY")

if required or optional:
    from kaggle_secrets import UserSecretsClient

    secrets = UserSecretsClient()
    for label in dict.fromkeys(required + optional):
        try:
            os.environ[label] = secrets.get_secret(label)
        except Exception:
            # Suppress backend exception details; they may contain sensitive response data.
            if label in required:
                raise RuntimeError(f"Enable the {label} secret for this notebook") from None
            os.environ.pop(label, None)
    del secrets
print("Required notebook secrets loaded: " + (", ".join(required) or "none for this config"))

## Check the runtime and prepare the shared validation corpus

Doctor checks actual operations on the selected GPU(s), plus required credential
presence. Successful secret loading does not prove permission to upload; verify the
smoke run's HF checkpoint and W&B run after training.

`prepare-eval` reuses an existing matching corpus. In a fresh session it can rebuild
from the pinned recipe. For research, pin `corpus_hash` in the shared evaluation
config to the printed fingerprint **before running the variants**. Alternatively,
preserve the prepared corpus as a separate attached dataset and point `corpus` at
its directory, then set `CORPUS_CONFIG = None`. Workspace sync excludes local `runs/`.
All variants and seeds must use the same corpus fingerprint and scoring profile.

In [ ]:
run_command(sys.executable, "-m", "nanoscope", "doctor", "--config", TRAIN_CONFIG)
if EVAL_CONFIG is not None:
    if CORPUS_CONFIG is not None:
        run_command(sys.executable, "-m", "nanoscope", "prepare-eval", "--config", CORPUS_CONFIG)
    # Verifies the exact corpus the evaluator will read, including a configured fingerprint.
    run_command(sys.executable, "-c",
                "import sys; from nanoscope.eval.config import load_eval_config; "
                "from nanoscope.eval.corpus import load_corpus; "
                "config=load_eval_config(sys.argv[1]); "
                "corpus=load_corpus(config.corpus, config.corpus_hash); "
                "print('Evaluation corpus:', corpus.path, 'fingerprint:', corpus.fingerprint)",
                EVAL_CONFIG)

## Train this variant and seed

The CLI starts the workers described by the training config. With two GPUs, both
train this one run; rank zero periodically scores the unwrapped model while the
other worker waits. Evaluation results and their history are included in checkpoint
uploads when HF persistence is enabled.

An interrupt caught by `run_command` sends SIGTERM to the CLI and waits for a graceful
stop. A forced kernel/session termination cannot guarantee a final save. Recover from
the last completed HF checkpoint. With `STOP_AFTER_STEP`, validation at that stop
step is deferred until resume; a very short interruption may therefore have no score.

In [ ]:
arguments = [sys.executable, "-m", "nanoscope", "train", "--config", TRAIN_CONFIG,
             "--resume", RESUME]
if EVAL_CONFIG is not None:
    arguments.extend(["--eval-config", EVAL_CONFIG])
if STOP_AFTER_STEP is not None:
    arguments.extend(["--stop-after-step", str(STOP_AFTER_STEP)])
run_command(*arguments)

## Download the result files for local comparison

Run this after training, or after a graceful interruption. The ZIP contains this
run's complete evaluation JSON files, training metrics and resolved config, plus
the workspace fingerprint. It deliberately excludes model weights and credentials.
Unzip it into your **local project root**: the `runs/<run-id>/evaluations/` layout
matches the study template. Repeat for each run, then run `nanoscope compare` locally.
HF checkpoints remain the recovery source if the session ends before you download.
There is no automatic Kaggle download or remote comparison in this notebook.

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
run_command(sys.executable, "-c", """
import json, os, sys, zipfile
from pathlib import Path
from nanoscope.config import load_config
from nanoscope.eval.runner import load_result
config = load_config(sys.argv[1])
run_dir = Path(config.run.output_dir) / config.run.id
if not run_dir.is_dir():
    raise RuntimeError('No run directory exists; train or resume first')
results = sorted((run_dir / 'evaluations').glob('*.json'))
for path in results:
    load_result(path)
destination = Path(sys.argv[2]) / (config.run.id + '-results.zip')
temporary = destination.with_suffix('.tmp')
try:
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        for path in results:
            archive.write(path, str(Path('runs') / config.run.id / 'evaluations' / path.name))
        for name in ('resolved-config.yaml', 'metrics.jsonl', 'evaluations.jsonl'):
            path = run_dir / name
            if path.is_file():
                archive.write(path, str(Path('runs') / config.run.id / name))
        archive.write('workspace-provenance.json',
                      str(Path('runs') / config.run.id / 'workspace-provenance.json'))
    os.replace(temporary, destination)
finally:
    temporary.unlink(missing_ok=True)
print(f'Exported {len(results)} complete evaluations to {destination}')
if not results:
    print('No evaluation scores yet. Resume with evaluation enabled before comparing.')
""", TRAIN_CONFIG, str(EXPORT_DIR))
print(f"Download the ZIP from {EXPORT_DIR} using the notebook output file browser.")

## Your next session or model increment

- **Same run, same session:** keep its config, set `RESUME = "auto"`, and clear
  `STOP_AFTER_STEP` to continue. Rerun the parameter cell, secret cell and training
  cell as appropriate. Installation is needed only once per session.
- **Same run, fresh session:** select the same configs, set
  `RESUME = "hf://OWNER/REPOSITORY/runs/RUN_ID"`, and run the notebook in order.
  Restore or rebuild the same corpus before training. Keep the same worker count,
  training recipe and compatible runtime. Resuming restores prior evaluation files
  from checkpoint history, so the export also includes earlier scores.
- **New increment or seed:** edit a training config locally, assign a new `run.id`
  and the intended `run.seed`, commit, and sync the workspace. Update the attached
  dataset version, select the new `TRAIN_CONFIG`, keep the shared corpus/evaluation
  settings, set `RESUME = "none"`, and rerun in order. The notebook is reused.

For your own M1 experiment, select e.g. `configs/my-base-seed-1337.yaml`,
`configs/my-corpus.yaml` and `configs/my-eval.yaml`. Copy these from the commented
experiment and evaluation templates, keeping Kaggle's stable training output path.
The fixture smoke config uses a different corpus/tokenizer from the research templates.

After downloading all variant/seed result ZIPs locally:

```bash
uv sync --extra eval
uv run nanoscope compare --study configs/my-study.yaml --output runs/my-comparison --plots
```

Update the study's run-ID globs and token budget. The default smoke run trains on
128 prediction tokens and produces one run; a comparison needs at least two variants.
A single seed per variant is exploratory; use identical seed sets across variants.

**Validation status:** notebook structure and orchestration are checked locally;
GPU execution and live Kaggle/HF/W&B integration still require your smoke run.